# Milestone 4, Section 1.4: Simulated drift and continual updates

This notebook runs the continual-learning experiment of Section 1. A stream of labelled users arrives in seven drift intervals, from clean data to the severe shift that broke the Milestone 2 model in Milestone 3 (all 12 raw features +3 sd), and then back to clean data. Four update strategies are compared, each with its own deployed model:

| Strategy | What happens when the trigger fires |
|---|---|
| `static` | Nothing: the starting model (v1-sim) stays deployed |
| `retrain_new` | Full retrain on the new interval only (Milestone 3 approach) |
| `retrain_replay` | **Selected:** full retrain on the new interval plus the replay buffer |
| `warm_replay` | Warm-start boosting from the deployed model on the new interval plus the replay buffer; old trees frozen |

The trigger and the promotion gate are the Milestone 3 rules: an update starts only when the deployed model's Qini is below the floor in two consecutive windows of the interval's gate rows, and a candidate replaces the deployed model only if the lower end of the 95% paired bootstrap interval of its Qini gain is above zero. Every metric uses the untouched Milestone 2 test split, under the interval's drift (current) and without drift (retention).

**Starting model.** The stream must consist of users the deployed model has never seen, otherwise the trigger and the promotion gate would check the model on its own training data, where its Qini looks better than it is. The Milestone 2 model v1 was trained on the whole training split, so the simulation starts from **v1-sim**: the same Milestone 2 configuration trained on the 2.8 million training rows that the stream does not use. The stream draws its 7 million rows from the rest.

The experiment is run with five seeds (42, 1, 2, 3, 4). The seed decides which training-split users fall into each interval, which rows the replay buffer keeps, and the bootstrap resamples; the test split, drift schedule and XGBoost settings stay the same.

- **Part 1** shows the main run (seed 42) in detail: tables, the four-panel figure, efficiency and saved model versions (Section 1.4 tables and Figure 1.1).
- **Part 2** summarizes all five seeds: mean ± sd of every metric, how often each strategy triggered and was promoted, update cost, and a paired bootstrap on the full test split comparing the selected strategy with the alternatives.

Run all cells (Runtime > Run all): about 50-70 minutes on a Colab CPU runtime. Set `SEEDS = (42,)` for the main run only (about 15 minutes). Results are written to `outputs/continual/`.

In [ ]:
# SETUP: install the XGBoost version the saved model was written with, and make the pipeline importable.
# The package is looked for (1) next to this notebook, (2) in an uploaded Milestone_4_code.zip,
# (3) in a clone of the GitHub repository (pulled again if it already exists).
%pip install -q xgboost==3.4.1 psutil
import os, sys, glob, zipfile, subprocess
PKG = None
if os.path.isdir("uplift_pipeline"):
    PKG = "."
if PKG is None and glob.glob("Milestone_4_code*.zip"):
    zipfile.ZipFile(sorted(glob.glob("Milestone_4_code*.zip"))[-1]).extractall(".")
    PKG = "Milestone 4"
if PKG is None:
    if os.path.isdir("INSE-6450"):
        subprocess.run(["git", "-C", "INSE-6450", "pull", "-q"], check=False)
    else:
        subprocess.run(["git", "clone", "-q", "https://github.com/saopa28/INSE-6450.git"], check=True)
    PKG = os.path.join("INSE-6450", "Milestone 4")
if not os.path.isdir(os.path.join(PKG, "uplift_pipeline")):
    raise FileNotFoundError(f"uplift_pipeline not found in {PKG!r}. Push the 'Milestone 4' folder to GitHub, "
                            "or upload Milestone_4_code.zip to the Colab file panel and run this cell again.")
sys.path.insert(0, os.path.abspath(PKG))
import xgboost; print("xgboost", xgboost.__version__, "| pipeline from", os.path.abspath(PKG))

In [ ]:
import json
import numpy as np, pandas as pd
from IPython.display import display
from uplift_pipeline import load_data, split, load_v1, summary, CONTINUAL, OUT_DIR
from uplift_pipeline.continual import fit_base, run_seeds, plot_results, plot_seeds, baseline_efficiency, STRATEGIES

FRAC = 1.0                     # share of rows to use; 1.0 reproduces the Milestone 2 split exactly
DATA = None                    # None = download the engineered data from the repository release
SEEDS = (42, 1, 2, 3, 4)       # the first seed is the main run shown in Part 1
OUT = OUT_DIR / "continual"
pd.set_option("display.width", 250)

df = load_data(frac=FRAC) if DATA is None else load_data(DATA, frac=FRAC)
sp = split(df); del df
print({k: f"{len(v[2]):,} rows" for k, v in sp.items()})
v1 = load_v1()                                  # Milestone 2 X-learner (its configuration is reused)
Xte, tte, yte = sp["test"]
print("v1 test metrics:", {k: round(v, 6) for k, v in summary(yte, tte, v1.score(Xte)).items()})
print("v1 trees used for prediction:", v1.trees())
base, stream_idx, base_idx = fit_base(v1, sp, out_dir=OUT)    # starting model v1-sim, trained on rows the stream never uses
print("v1-sim trees used for prediction:", base.trees())
pd.DataFrame([{"hyperparameter": k, "value": v} for k, v in CONTINUAL.items()])

In [ ]:
# Runs the simulation once per seed (each seed's results and models go to outputs/continual/seed_<n>/)
res_all, summ, upd_all, upd_summ, boots = run_seeds(base, sp, seeds=SEEDS, out_dir=OUT, stream_idx=stream_idx,
                                                    base_idx=base_idx, base_version="v1-sim")
MAIN = SEEDS[0]
res, upd = res_all[res_all.seed == MAIN].drop(columns="seed"), upd_all[upd_all.seed == MAIN].drop(columns="seed")

# Part 1: main run (seed 42)
## Metrics before and after each update
`qini_before` is the deployed model when the interval arrives; `qini_after` is the model deployed after the interval's update decision. Both are on the test split under the interval's drift. `qini_original` is the deployed model on the test split without drift (retention of old knowledge).

In [ ]:
cols = ["step", "drift", "strategy", "psi_max", "monitor", "triggered", "updated", "promoted", "version_after",
        "qini_before", "qini_after", "auuc_before", "auuc_after", "uplift10_before", "uplift10_after", "qini_original", "auuc_original", "gate_ci_lo", "gate_ci_hi"]
display(res.reindex(columns=cols).round(6))

In [ ]:
cur = res.pivot(index=["step", "drift"], columns="strategy", values="qini_after")[list(STRATEGIES)]
ret = res.pivot(index=["step", "drift"], columns="strategy", values="qini_original")[list(STRATEGIES)]
print("Qini under the current drift"); display(cur.round(6))
print("Qini on the original (undrifted) test data"); display(ret.round(6))

## Efficiency of each promoted update
Update time, peak process memory (RSS) during the update, model size (stage-2 boosters used for scoring), number of trees, and model-only scoring latency for one request (batch 1) and a 256-row batch. Colab CPU runtime, so there is no VRAM figure.

In [ ]:
eff = pd.concat([pd.DataFrame([baseline_efficiency(base, Xte, "v1-sim")]), upd], ignore_index=True)
eff.to_csv(OUT / f"seed_{MAIN}" / "efficiency.csv", index=False)
display(eff.round(3))

In [ ]:
fig = plot_results(res, upd, OUT / f"seed_{MAIN}")       # Figure 1.1

In [ ]:
# Saved model versions of the main run (each has a manifest with its parent, training rows, gate result and file hashes)
for d in sorted((OUT / f"seed_{MAIN}" / "models").iterdir()):
    m = json.loads((d / "manifest.json").read_text())
    print(f"{m['version']:24s} parent={m['parent']:22s} trees={m['trees']} size={m['scoring_size_mb']} MB")

# Part 2: all seeds
## Mean ± sd across seeds

In [ ]:
def table(metric):
    m = summ.pivot(index=["step", "drift"], columns="strategy", values=f"{metric}_mean")[list(STRATEGIES)]
    s = summ.pivot(index=["step", "drift"], columns="strategy", values=f"{metric}_sd")[list(STRATEGIES)]
    return m.map(lambda v: f"{v:.6f}") + " ± " + s.fillna(0).map(lambda v: f"{v:.6f}")
print("Qini under the current drift"); display(table("qini_after"))
print("Qini on the original test data (retention)"); display(table("qini_original"))
print("Uplift@10% under the current drift"); display(table("uplift10_after"))
print(f"Seeds (out of {len(SEEDS)}) that triggered / promoted an update")
display(summ.pivot(index=["step", "drift"], columns="strategy", values="seeds_triggered")[list(STRATEGIES)].astype(int).astype(str)
        + " / " + summ.pivot(index=["step", "drift"], columns="strategy", values="seeds_promoted")[list(STRATEGIES)].astype(int).astype(str))

## Update cost across seeds

In [ ]:
display(upd_summ.round(3))

## Paired bootstrap on the test split: selected strategy minus the alternative
A positive gain means retrain + replay ranks better. The interval is the 95% paired bootstrap interval on the full test split (200 resamples); a gain of 0 with an empty interval means both strategies had the same model deployed.

In [ ]:
display(boots.round(6))
print("\nSeeds where the selected strategy is significantly better (CI above 0) / worse (CI below 0):")
display(boots.groupby(["step", "drift", "other"]).apply(
    lambda g: pd.Series({"better": int((g.ci_lo > 0).sum()), "worse": int((g.ci_hi < 0).sum()),
                         "mean_gain": g.qini_gain.mean()}), include_groups=False))

In [ ]:
fig = plot_seeds(summ, OUT)
print("Files written:", sorted(p.name for p in OUT.iterdir()))